In [1]:
import os

In [2]:
os.chdir("../")

#### Update the Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int
    

#### Update the Configuration Manager

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        model_trainer_config = self.config.model_trainer

        create_directories([model_trainer_config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir = Path(model_trainer_config.root_dir),
            data_path = Path(model_trainer_config.data_path),
            model_ckpt = model_trainer_config.model_ckpt,
            num_train_epochs = self.params.TrainingArguments.num_train_epochs,
            warmup_steps = self.params.TrainingArguments.warmup_steps,
            per_device_train_batch_size = self.params.TrainingArguments.per_device_train_batch_size,
            weight_decay = self.params.TrainingArguments.weight_decay,
            logging_steps = self.params.TrainingArguments.logging_steps,
            eval_strategy = self.params.TrainingArguments.eval_strategy,
            eval_steps = self.params.TrainingArguments.eval_steps,
            save_steps = self.params.TrainingArguments.save_steps,
            gradient_accumulation_steps = self.params.TrainingArguments.gradient_accumulation_steps
        )

        return model_trainer_config

#### Update the Components

In [6]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk
import torch

[2026-02-25 15:50:35,154: INFO: config]: TensorFlow version 2.20.0 available.


In [7]:
class ModelTrainer:

    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        # loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=self.config.num_train_epochs, 
            warmup_steps=self.config.warmup_steps, 
            per_device_train_batch_size=self.config.per_device_train_batch_size, 
            per_device_eval_batch_size=self.config.per_device_train_batch_size, 
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
            )
        
        # trainer_args = TrainingArguments(
        #     output_dir = 'pegasus-samsum', num_train_epochs = 1, warmup_steps = 500,
        #     per_device_train_batch_size =1, per_device_eval_batch_size = 1,
        #     weight_decay=0.01, logging_steps = 10,
        #     eval_steps = 500, save_steps = 1e6,
        #     gradient_accumulation_steps = 16
        # )

        trainer = Trainer(model = model_pegasus, args = trainer_args, 
                          train_dataset = dataset_samsum_pt["test"],
                          eval_dataset = dataset_samsum_pt["validation"],
                          data_collator=seq2seq_data_collator)
        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

#### Update the Pipeline

In [8]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-02-25 15:50:37,828: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


[2026-02-25 15:50:37,829: WARNING: _http]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-02-25 15:50:37,897: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-02-25 15:50:38,238: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-02-25 15:50:38,304: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-02-25 15:50:38,613: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Fou

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[2026-02-25 15:50:43,162: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/xet-read-token/7db4e727842ad86cc5cdfeaef4d14c6fb2b46511 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

...

'_ssl.c:999: The handshake operation timed out' thrown while requesting GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/xet-read-token/7db4e727842ad86cc5cdfeaef4d14c6fb2b46511


[2026-02-25 16:12:15,487: WARNING: _http]: '_ssl.c:999: The handshake operation timed out' thrown while requesting GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/xet-read-token/7db4e727842ad86cc5cdfeaef4d14c6fb2b46511


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\FPK1COB\Documents\Learning\ML-assignments\ML-Assignments\venv\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\FPK1COB\Documents\Learning\ML-assignments\ML-Assignments\venv\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\FPK1COB\Documents\Learning\ML-assignments\ML-Assignments\venv\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\FPK1COB\Documents\Learning\ML-assignments\ML-Assignments\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "c:\Users\FPK1COB\Documents\Learning\ML-assignments\ML-Assignments\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 113, in auto_conversion
    resolved_archive_file = cached_file(pretrained_model_name_or_path, filename, **cached_fi

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]